# Part 3 MLflow Experiment Tracking and Model Versioning

This notebook applies MLflow as the advanced AI and MLOps technique for the smart-city traffic solution. MLflow was selected because the project contains multiple modelling approaches with different parameters, evaluation metrics and saved artifacts. Recording these elements within a consistent experiment makes the results easier to compare, reproduce and audit.

The neural network and the comparable Random Forest developed for the SHAP analysis are tracked as successive solution candidates. Each candidate is assigned an explicit solution version, model version and candidate status. Its parameters, evaluation metrics and available artifacts are then recorded using MLflow.

Within this specific comparison, the neural network is labelled as the baseline and the Random Forest is labelled as the selected candidate because it achieved stronger test performance. The term `selected_candidate` refers only to the better-performing model within this MLflow experiment. It does not indicate that the SHAP Random Forest supersedes the tuned Random Forest identified as the strongest overall predictive model in the earlier modelling analysis.

A local SQLite tracking database is used because it provides a structured and persistent MLflow backend without requiring an external tracking server. This implementation supports evidence-based comparison and provides a foundation for future deployment, governance and monitoring.

In [1]:
import sys

print("Executable:", sys.executable)
print("Python:", sys.version)

Executable: /usr/local/bin/python3
Python: 3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]


In [ ]:
%pip install "mlflow>=2.10,<4.0"

In [2]:
import mlflow

print("MLflow:", mlflow.__version__)

MLflow: 3.16.1


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path

import mlflow
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = Path.cwd()

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MLFLOW_DIR = PROJECT_ROOT / "mlflow"

MLFLOW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACTS_DIR = MLFLOW_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRACKING_DATABASE = MLFLOW_DIR / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{TRACKING_DATABASE.resolve()}"
)

EXPERIMENT_NAME = "smart_city_traffic_demand"

client = mlflow.MlflowClient()

experiment = client.get_experiment_by_name(
    EXPERIMENT_NAME
)

if experiment is None:
    client.create_experiment(
        EXPERIMENT_NAME,
        artifact_location=ARTIFACTS_DIR.resolve().as_uri(),
    )

mlflow.set_experiment(
    EXPERIMENT_NAME
)

COMPARISON_PATH = (
    REPORTS_DIR
    / "deep_learning_model_comparison.csv"
)

comparison_df = pd.read_csv(
    COMPARISON_PATH
)

print("MLflow version:", mlflow.__version__)
print("Tracking database:", TRACKING_DATABASE)
print("Artifact location:", ARTIFACTS_DIR)
print("Experiment:", EXPERIMENT_NAME)

comparison_df.round(4)

2026/09/25 03:43:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/25 03:43:06 INFO mlflow.store.db.utils: Updating database tables


MLflow version: 3.16.1
Tracking database: /Users/misatoniwa/Downloads/smart-city-traffic-capstone/part3_machine_learning/mlflow/mlflow.db
Artifact location: /Users/misatoniwa/Downloads/smart-city-traffic-capstone/part3_machine_learning/mlflow/artifacts
Experiment: smart_city_traffic_demand


,Model,MAE,R2
0,Neural Network,493.0636,0.8845
1,Random Forest for SHAP,295.5493,0.9277


## Logging model experiments

The neural network and comparable Random Forest results are recorded as separate MLflow runs. For each run, the solution version, model version, candidate status, model parameters, evaluation metrics and available artifacts are logged to support reproducibility and consistent comparison.

Both runs use results obtained from the same 23 input features and the same chronologically held-out 20% test period. This ensures that differences in performance are not caused by different test samples.

In [ ]:
def log_artifact_if_available(file_path, artifact_path):
    if file_path.exists():
        mlflow.log_artifact(
            str(file_path),
            artifact_path=artifact_path,
        )


model_metrics = comparison_df.set_index("Model")

with mlflow.start_run(
    run_name="neural_network"
):
    mlflow.set_tags(
        {
            "solution_version": "v1",
            "model_version": "neural_network_v1",
            "candidate_status": "baseline",
            "data_version": "traffic_features_v1",
        }
    )

    mlflow.log_params(
        {
            "model_type": "feed_forward_neural_network",
            "number_of_features": 23,
            "hidden_layers": "64,32",
            "dropout_rates": "0.20,0.10",
            "test_split": 0.20,
            "split_method": "chronological",
        }
    )

    mlflow.log_metrics(
        {
            "mae": float(
                model_metrics.loc[
                    "Neural Network",
                    "MAE",
                ]
            ),
            "r2": float(
                model_metrics.loc[
                    "Neural Network",
                    "R2",
                ]
            ),
        }
    )

    log_artifact_if_available(
        MODELS_DIR / "traffic_neural_network.keras",
        "model",
    )
    log_artifact_if_available(
        MODELS_DIR / "neural_network_x_scaler.joblib",
        "preprocessing",
    )
    log_artifact_if_available(
        MODELS_DIR / "neural_network_y_scaler.joblib",
        "preprocessing",
    )
    log_artifact_if_available(
        REPORTS_DIR / "neural_network_training_loss.png",
        "figures",
    )
    log_artifact_if_available(
        REPORTS_DIR / "neural_network_results.csv",
        "results",
    )


with mlflow.start_run(
    run_name="random_forest_shap"
):
    mlflow.set_tags(
        {
            "solution_version": "v2",
            "model_version": "random_forest_shap_v1",
            "candidate_status": "selected_candidate",
            "data_version": "traffic_features_v1",
        }
    )
    
    mlflow.log_params(
        {
            "model_type": "random_forest_regressor",
            "number_of_features": 23,
            "n_estimators": 150,
            "max_depth": 18,
            "min_samples_leaf": 2,
            "random_state": 42,
            "test_split": 0.20,
            "split_method": "chronological",
        }
    )

    mlflow.log_metrics(
        {
            "mae": float(
                model_metrics.loc[
                    "Random Forest for SHAP",
                    "MAE",
                ]
            ),
            "r2": float(
                model_metrics.loc[
                    "Random Forest for SHAP",
                    "R2",
                ]
            ),
        }
    )

    log_artifact_if_available(
        MODELS_DIR / "shap_random_forest.joblib",
        "model",
    )
    log_artifact_if_available(
        REPORTS_DIR / "shap_feature_summary.png",
        "figures",
    )
    log_artifact_if_available(
        REPORTS_DIR / "shap_feature_importance.csv",
        "results",
    )

print("Successfully logged both model runs to MLflow.")

Successfully logged both model runs to MLflow.


## MLflow run comparison

The recorded MLflow runs are retrieved from the local tracking database to confirm that their parameters, tags, evaluation metrics and completion statuses were stored successfully. The comparison table provides an auditable summary of the two solution versions and verifies that both runs completed successfully.

In [5]:
experiment = mlflow.get_experiment_by_name(
    EXPERIMENT_NAME
)

runs_df = mlflow.search_runs(
    experiment_ids=[
        experiment.experiment_id
    ],
    order_by=[
        "start_time DESC",
    ],
)

versioned_runs = runs_df[
    runs_df[
        "tags.model_version"
    ].notna()
].copy()

versioned_runs = (
    versioned_runs.drop_duplicates(
        subset=[
            "tags.model_version",
        ],
        keep="first",
    )
)

run_comparison = versioned_runs[
    [
        "tags.solution_version",
        "tags.model_version",
        "tags.candidate_status",
        "tags.data_version",
        "tags.mlflow.runName",
        "metrics.mae",
        "metrics.r2",
        "params.model_type",
        "params.split_method",
        "status",
    ]
].rename(
    columns={
        "tags.solution_version": (
            "Solution_Version"
        ),
        "tags.model_version": (
            "Model_Version"
        ),
        "tags.candidate_status": (
            "Candidate_Status"
        ),
        "tags.data_version": (
            "Data_Version"
        ),
        "tags.mlflow.runName": "Run",
        "metrics.mae": "MAE",
        "metrics.r2": "R2",
        "params.model_type": "Model_Type",
        "params.split_method": "Split_Method",
        "status": "Status",
    }
)

run_comparison = (
    run_comparison.sort_values(
        "Solution_Version"
    )
    .reset_index(drop=True)
)

run_comparison.round(4)

,Solution_Version,Model_Version,Candidate_Status,Data_Version,Run,MAE,R2,Model_Type,Split_Method,Status
0,v1,neural_network_v1,baseline,traffic_features_v1,neural_network,493.0636,0.8845,feed_forward_neural_network,chronological,FINISHED
1,v2,random_forest_shap_v1,selected_candidate,traffic_features_v1,random_forest_shap,295.5493,0.9277,random_forest_regressor,chronological,FINISHED


## Interpretation, value and limitations

MLflow successfully recorded the neural network as solution version `v1` and the comparable Random Forest as solution version `v2`. Each run contains its model version, data version, candidate status, parameters, evaluation metrics and available saved artifacts.

Within this MLflow experiment, the Random Forest was labelled as the selected candidate because it achieved a lower mean absolute error of approximately 296 vehicles per hour and a higher R-squared value of 0.9277. The neural-network baseline achieved a mean absolute error of approximately 493 vehicles per hour and an R-squared value of 0.8845. Both models were evaluated using the same chronologically held-out test period, allowing a consistent comparison.

The `selected_candidate` label identifies the stronger model within this specific comparison between the neural network and the Random Forest used for SHAP analysis. It does not represent a replacement for the tuned Random Forest identified as the strongest overall predictive model in the earlier modelling analysis. Instead, the label records the decision made within the scope of this MLflow experiment.

MLflow adds value by storing model-development evidence within a structured experiment rather than relying on manually recorded results. It improves reproducibility, supports comparisons between solution versions and creates an auditable connection between model settings, performance metrics and artifacts. This would make it easier to investigate, review or replace a deployed model in the future.

The implementation has several limitations. The tracking database and artifacts are stored locally, so they are not automatically shared across users or machines. Version and candidate-status tags are assigned manually and could become inconsistent without a defined governance process. The notebook records completed model artifacts but does not provide automated retraining, approval workflows or a remote production model registry. A production implementation would require shared storage, access controls, automated validation, ongoing performance monitoring and formal promotion criteria.